In [ ]:
import sys
print(sys.executable)

In [3]:
import boto3
import pandas as pd
import pyarrow.parquet as pq
import io

s3 = boto3.client("s3", region_name="us-east-2")

BUCKET = "hmda-funnel-analytics-juanjo"

In [5]:
cols_a_chequear = ["property_value", "interest_rate", "rate_spread",
                   "total_loan_costs", "origination_charges", "discount_points"]

objs = s3.list_objects_v2(Bucket=BUCKET, Prefix="raw/").get("Contents", [])

tipos_por_estado = {}
for obj in objs:
    key = obj["Key"]
    state = key.split("hmda_")[1].replace(".parquet", "")
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    schema = pq.read_schema(io.BytesIO(body))
    tipos_por_estado[state] = {col: str(schema.field(col).type) for col in cols_a_chequear if col in schema.names}

tipos_df = pd.DataFrame(tipos_por_estado).T
tipos_df

,property_value,interest_rate,rate_spread,total_loan_costs,origination_charges,discount_points
AZ,string,string,string,string,string,string
CA,string,string,string,string,string,string
FL,string,string,string,string,string,string
GA,string,string,string,string,string,string
IL,string,string,string,string,string,string
IN,string,string,string,string,string,string
MA,string,string,string,string,string,string
MD,string,string,string,string,string,string
MI,string,string,string,string,string,string
MO,string,string,string,string,string,string


In [7]:
import pyarrow.parquet as pq
import io
import pandas as pd

BUCKET = "hmda-funnel-analytics-juanjo"
objs = s3.list_objects_v2(Bucket=BUCKET, Prefix="raw/").get("Contents", [])

todas_las_cols = [
    "activity_year","action_taken","denial_reason-1","preapproval",
    "loan_amount","property_value","interest_rate","rate_spread",
    "discount_points","total_loan_costs","origination_charges",
    "derived_loan_product_type","derived_dwelling_category","conforming_loan_limit",
    "derived_ethnicity","derived_race","derived_sex","applicant_age","income",
    "debt_to_income_ratio","state_code","county_code",
    "ffiec_msa_md_median_family_income",
]

tipos_por_estado = {}
for obj in objs:
    key = obj["Key"]
    state = key.split("hmda_")[1].replace(".parquet", "")
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
    schema = pq.read_schema(io.BytesIO(body))
    tipos_por_estado[state] = {col: str(schema.field(col).type) for col in todas_las_cols if col in schema.names}

tipos_df = pd.DataFrame(tipos_por_estado).T

# Lo que declaramos en Athena (el DROP TABLE / CREATE anterior)
declarado_athena = {
    "activity_year": "int32/int64", "action_taken": "int32/int64",
    "denial_reason-1": "string", "preapproval": "int32/int64",
    "loan_amount": "double", "property_value": "string", "interest_rate": "string",
    "rate_spread": "string", "discount_points": "string", "total_loan_costs": "string",
    "origination_charges": "string", "derived_loan_product_type": "string",
    "derived_dwelling_category": "string", "conforming_loan_limit": "string",
    "derived_ethnicity": "string", "derived_race": "string", "derived_sex": "string",
    "applicant_age": "string", "income": "string", "debt_to_income_ratio": "string",
    "state_code": "string", "county_code": "string",
    "ffiec_msa_md_median_family_income": "double",
}

# Chequeo 1: ¿todos los estados coinciden entre sí, columna por columna?
print("=== Columnas con tipo INCONSISTENTE entre estados ===")
for col in todas_las_cols:
    tipos_unicos = tipos_df[col].unique()
    if len(tipos_unicos) > 1:
        print(f"{col}: {dict(tipos_df[col].value_counts())}")

print("\n=== Tipo real (pandas/parquet) vs. declarado en Athena ===")
resumen = pd.DataFrame({
    "tipo_real_parquet": tipos_df.iloc[0],
    "declarado_en_athena": pd.Series(declarado_athena)
})
resumen["MISMATCH"] = ~resumen.apply(
    lambda r: (("string" in str(r["tipo_real_parquet"])) == ("string" in str(r["declarado_en_athena"]))), axis=1
)
resumen

=== Columnas con tipo INCONSISTENTE entre estados ===

=== Tipo real (pandas/parquet) vs. declarado en Athena ===


,tipo_real_parquet,declarado_en_athena,MISMATCH
activity_year,int64,int32/int64,False
action_taken,int64,int32/int64,False
denial_reason-1,int64,string,True
preapproval,int64,int32/int64,False
loan_amount,double,double,False
property_value,string,string,False
interest_rate,string,string,False
rate_spread,string,string,False
discount_points,string,string,False
total_loan_costs,string,string,False
